# Step Independence Analysis -- Fermi (RealFP)

Same analysis as [step_independence.ipynb](step_independence.ipynb) (GAIA), applied to the Fermi
naive ReAct and MarkovReAct trajectories instead. The judge logic is dataset-agnostic -- it only
looks at each step's `model_output`/`code_action` -- so it's reused unchanged; only MODEL_DIRS and
the output filenames differ, to avoid colliding with the GAIA `redundant_pairs_*.json` files.

Ask an LLM judge whether each consecutive pair of steps in a ReAct trajectory has Step B re-attempting the same action Step A already attempted (e.g. a reworded repeat search, re-visiting the same page, retrying an equivalent approach), as opposed to Step B building on Step A's result to take a genuinely new action -- even one on the same topic or source.

In [1]:
import os
import pickle
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
judge_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

JUDGE_MODEL = "gpt-4o-mini"

In [2]:
JUDGE_SYSTEM_PROMPT = """You are evaluating one AI agent's step-by-step trajectory while solving a task.
You will be shown the task, then two consecutive steps (each a "thought" followed by a code snippet) taken by the agent.
Decide whether Step B is RE-ATTEMPTING the same action Step A already attempted -- e.g. repeating a similar search
query, re-visiting the same page, re-extracting the same piece of information, retrying an equivalent approach --
as opposed to Step B using or building on what Step A did/found to take a NEW action, even if that new action
concerns the same topic, source, or document as Step A.
Answer with exactly one word: "yes" or "no"."""


def format_step(step: dict) -> str:
    return step.get("model_output") or ""


def same_subgoal(question: str, step_a: dict, step_b: dict, model: str = JUDGE_MODEL) -> bool:
    """Ask the judge model whether step_b is re-attempting the same action as step_a."""
    user_content = (
        f"Task: {question}\n\n"
        f"=== Step A ===\n{format_step(step_a)}\n\n"
        f"=== Step B ===\n{format_step(step_b)}"
    )
    response = judge_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        temperature=0,
    )
    verdict = response.choices[0].message.content.strip().lower()
    return verdict.startswith("yes")

In [3]:
import json
import re
from concurrent.futures import ThreadPoolExecutor

KNOWN_TOOLS = [
    "web_search", "visit_page", "page_up", "page_down", "find_on_page_ctrl_f",
    "find_next", "find_archived_url", "inspect_file_as_text", "visualizer", "final_answer",
]


def tools_in_step(step: dict) -> set[str]:
    code = step.get("code_action") or ""
    return {t for t in KNOWN_TOOLS if re.search(rf"\b{re.escape(t)}\s*\(", code)}


def step_details(step: dict) -> dict:
    return {
        "step_number": step.get("step_number"),
        "model_output": step.get("model_output"),
        "code_action": step.get("code_action"),
        "observations": step.get("observations"),
        "token_usage": step.get("token_usage"),
    }


def scan_trajectory(traj: dict, executor: ThreadPoolExecutor) -> list[dict]:
    """Run the judge over every consecutive pair in one trajectory (concurrently); return only the
    redundant ones, with full step details included so findings can be written out and inspected later."""
    steps = traj["steps"]
    pairs = list(range(len(steps) - 1))
    verdicts = executor.map(lambda i: same_subgoal(traj["question"], steps[i], steps[i + 1]), pairs)

    findings = []
    for i, verdict in zip(pairs, verdicts):
        if verdict:
            findings.append({
                "task_id": traj.get("task_id", "?"),
                "question": traj["question"],
                "step_i": i,
                "step_j": i + 1,
                "tools": sorted(tools_in_step(steps[i]) | tools_in_step(steps[i + 1])),
                "steps": {
                    str(i): step_details(steps[i]),
                    str(i + 1): step_details(steps[i + 1]),
                },
            })
    return findings

In [4]:
BASELINE_DIR = Path("../baseline")
MARKOV_DIR = Path("../markovReAct")

MODEL_DIRS = {
    # naive baselines
    "gpt-4o": BASELINE_DIR / "naive_fermi_gpt-4o",
    "gpt-5.4-mini": BASELINE_DIR / "naive_fermi_gpt-5.4-mini",
    "Qwen3.7-Plus": BASELINE_DIR / "naive_fermi_Qwen_Qwen3.7-Plus",
    "Qwen3.5-9B": BASELINE_DIR / "naive_fermi_Qwen_Qwen3.5-9B",
    # markovReAct, window_size in {1, 3, 5}
    "gpt-4o_w1": MARKOV_DIR / "markov_fermi_gpt-4o_w1",
    "gpt-4o_w3": MARKOV_DIR / "markov_fermi_gpt-4o_w3",
    "gpt-4o_w5": MARKOV_DIR / "markov_fermi_gpt-4o_w5",
    "gpt-5.4-mini_w1": MARKOV_DIR / "markov_fermi_gpt-5.4-mini_w1",
    "gpt-5.4-mini_w3": MARKOV_DIR / "markov_fermi_gpt-5.4-mini_w3",
    "gpt-5.4-mini_w5": MARKOV_DIR / "markov_fermi_gpt-5.4-mini_w5",
    "Qwen3.7-Plus_w1": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.7-Plus_w1",
    "Qwen3.7-Plus_w3": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.7-Plus_w3",
    "Qwen3.7-Plus_w5": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.7-Plus_w5",
    "Qwen3.5-9B_w1": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.5-9B_w1",
    "Qwen3.5-9B_w3": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.5-9B_w3",
    "Qwen3.5-9B_w5": MARKOV_DIR / "markov_fermi_Qwen_Qwen3.5-9B_w5",
}


def load_cache(output_path: Path) -> tuple[set[str], list[dict]]:
    """Load {scanned_task_ids, findings} from output_path if present."""
    if not output_path.exists():
        return set(), []
    with open(output_path) as f:
        data = json.load(f)
    return set(data.get("scanned_task_ids", [])), data.get("findings", [])


results_by_model = {}
with ThreadPoolExecutor(max_workers=10) as executor:
    for model_name, pkl_dir in MODEL_DIRS.items():
        output_path = Path(f"redundant_pairs_fermi_{model_name}.json")
        scanned_task_ids, all_findings = load_cache(output_path)
        if scanned_task_ids:
            print(f"{model_name}: resuming, {len(scanned_task_ids)} trajectory(ies) already cached")

        for pkl_file in sorted(pkl_dir.glob("*.pkl"), key=lambda p: int(p.stem)):
            with open(pkl_file, "rb") as f:
                traj = pickle.load(f)
            if "steps" not in traj:
                continue
            task_id = traj.get("task_id", "?")
            if task_id in scanned_task_ids:
                continue

            all_findings.extend(scan_trajectory(traj, executor))
            scanned_task_ids.add(task_id)

            # Persist after every trajectory so an interrupted run only loses in-flight work,
            # not everything scanned so far.
            with open(output_path, "w") as f:
                json.dump(
                    {"scanned_task_ids": sorted(scanned_task_ids), "findings": all_findings},
                    f, indent=2, default=str,
                )

        results_by_model[model_name] = all_findings
        print(f"{model_name}: {len(all_findings)} redundant pair(s) across {len(scanned_task_ids)} trajectories -> {output_path}")

gpt-4o: 130 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_gpt-4o.json


gpt-5.4-mini: 32 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_gpt-5.4-mini.json


Qwen3.7-Plus: 324 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_Qwen3.7-Plus.json


Qwen3.5-9B: 228 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_Qwen3.5-9B.json


gpt-4o_w1: 89 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_gpt-4o_w1.json


...

Qwen3.7-Plus_w3: 568 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_Qwen3.7-Plus_w3.json


Qwen3.7-Plus_w5: 426 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_Qwen3.7-Plus_w5.json


Qwen3.5-9B_w1: 995 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_Qwen3.5-9B_w1.json


Qwen3.5-9B_w3: 510 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_Qwen3.5-9B_w3.json


Qwen3.5-9B_w5: 363 redundant pair(s) across 125 trajectories -> redundant_pairs_fermi_Qwen3.5-9B_w5.json


In [5]:
def total_pairs(pkl_dir: Path) -> int:
    """Total number of consecutive step-pairs across every trajectory in pkl_dir."""
    total = 0
    for pkl_file in pkl_dir.glob("*.pkl"):
        with open(pkl_file, "rb") as f:
            traj = pickle.load(f)
        total += max(0, traj.get("num_steps", 0) - 1)
    return total


rows = []
for model_name, pkl_dir in MODEL_DIRS.items():
    output_path = Path(f"redundant_pairs_fermi_{model_name}.json")
    if not output_path.exists():
        continue
    with open(output_path) as f:
        data = json.load(f)
    n_redundant = len(data["findings"])
    n_total = total_pairs(pkl_dir)
    rows.append({
        "config": model_name,
        "redundant_pairs": n_redundant,
        "total_pairs": n_total,
        "fraction_redundant": n_redundant / n_total if n_total else None,
    })

norm_df = pd.DataFrame(rows)
norm_df["fraction_redundant_pct"] = norm_df["fraction_redundant"].map(lambda x: f"{x:.1%}" if x is not None else "-")
norm_df

,config,redundant_pairs,total_pairs,fraction_redundant,fraction_redundant_pct
0,gpt-4o,130,519,0.250482,25.0%
1,gpt-5.4-mini,32,153,0.209150,20.9%
2,Qwen3.7-Plus,324,1885,0.171883,17.2%
3,Qwen3.5-9B,228,1315,0.173384,17.3%
4,gpt-4o_w1,89,481,0.185031,18.5%
5,gpt-4o_w3,93,403,0.230769,23.1%
6,gpt-4o_w5,78,436,0.178899,17.9%
7,gpt-5.4-mini_w1,31,154,0.201299,20.1%
8,gpt-5.4-mini_w3,36,150,0.240000,24.0%
9,gpt-5.4-mini_w5,32,150,0.213333,21.3%
